Ejercicio M4 C5: **Diseño de un agente de soporte técnico autónomo y seguro**

Instrucciones:

Basándose en el stack tecnológico recomendado (LangGraph, Docker, y Bases Vectoriales), deberán esquematizar el flujo de un agente que resuelva el siguiente caso: "Un cliente solicita el reembolso de una factura debido a un error en el sistema de pagos".

Tareas a realizar:



1.   Definición de Percepción y Herramientas: Identifique qué APIs necesitaría el agente (ej. acceso a SQL de facturación, pasarela de pagos).
2.   Diseño del Ciclo ReAct: Redacte el "Protocolo de Pensamiento" que el agente debe seguir (Análisis →Planificación →Pensamiento → Acción → Observación).
3. Implementación de Seguridad:
    * Diseñe un nodo de "Human-in-the-loop" para la fase de ejecución del reembolso.
    * Establezca un Sandboxing de Ejecución mediante Docker para que el agente pruebe el cálculo del reembolso antes de enviarlo a la API real.
4. Auditoría ISO 42001: Cree un esquema de Log de Evidencias (JSON) que registre la trazabilidad del pensamiento del agente para una futura auditoría de transparencia.



# 🎓 Agente de Reintegros

**Flujo:** el usuario solicita un reintegro → el agente busca en la base de facturación si
corresponde → ejecuta las devoluciones que puede ejecutar solo y **reserva al humano** las que no.

## Dos fuentes de datos

| Dataset | Qué es | Quién lo escribe |
|---|---|---|
| `solicitudes.csv` | Lo que **pide** el usuario, en lenguaje natural | El cliente. **No es confiable.** |
| `facturas.csv` | Lo que **dice el sistema de pagos** | La empresa. Es la fuente de verdad. |


## Niveles de autoridad del agente

| Nivel | Caso | Quién actúa |
|---|---|---|
| 1 — ejecuta solo | `cobro_duplicado` dentro del límite, y el solicitante es el titular | El agente mueve el dinero |
| 2 — responde solo | `pago_correcto` | El agente informa; no mueve dinero |
| 3 — reservado | Todo lo demás: estado no automatizable, monto sobre el límite, factura inexistente, titular que no coincide, o desacuerdo entre el LLM y la política | El grafo se **pausa** y espera a una persona |

## Limitaciones

El LLM interpreta el pedido en lenguaje natural, propone la
acción, y redacta la respuesta.

El LLM **no** tiene la última palabra sobre mover dinero. Su decisión se contrasta contra la
política (`verificar_decision`). Si coinciden, se ejecuta; si no, sube a un humano. Es decir:
**el LLM puede frenar una devolución que no corresponde, pero no puede autorizar una que el registro no respalda.**
Toda duda cae del lado de la persona, nunca del lado de ejecutar.


## Paso 1: Instalación de Ollama

In [ ]:
# Instalamos Ollama (el motor que corre el modelo localmente)
!sudo apt-get update && sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

!pip install -q langgraph langchain-core requests
print("Dependencias instaladas.")

## Paso 2: Levantar el servidor de Ollama y descargar el modelo

Ollama funciona como un servidor: lo prendemos en segundo plano (`subprocess.Popen`) y
después le pedimos que descargue `phi3:mini`. Puede tardar 1-2 minutos la primera vez.

In [ ]:
import subprocess, time, requests

subprocess.run(["pkill", "ollama"], stderr=subprocess.DEVNULL)
time.sleep(2)
with open("ollama.log", "w") as log_file:
    subprocess.Popen(["ollama", "serve"], stdout=log_file, stderr=log_file)
time.sleep(5)

!ollama pull phi3:mini

print("Ollama listo con el modelo phi3:mini.")

## Paso 3: Función para hablar con Ollama

Si Ollama falla, la función devuelve un texto de error.

In [ ]:
import requests

MODELO = "phi3:mini"

def consultar_ollama(prompt: str, model: str = MODELO, temperatura: float = 0.0) -> str:
    """Envía un prompt a Ollama y devuelve el texto de la respuesta (o un texto de error)."""
    url = "http://localhost:11434/api/generate"
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False,
        "options": {"temperature": temperatura},
    }
    try:
        respuesta = requests.post(url, json=payload, timeout=180)
        respuesta.raise_for_status()
        return respuesta.json()["response"].strip()
    except Exception as e:
        return f"__ERROR_LLM__: {type(e).__name__}: {e}"

print(consultar_ollama("Respondé en una sola palabra: ¿2+2 es par o impar?"))

## Paso 4 — Dataset A: la base de facturación (sistema de registro)

Lo que el sistema de pagos sabe de cada factura. Es la fuente de verdad contra la que el
agente verifica.

In [ ]:
import pandas as pd
import os

RUTAS_FACTURAS = ["/content/facturas2.csv", "/content/facturas.csv",
                  "datos/facturas.csv", "facturas2.csv", "facturas.csv"]
CSV_FACTURAS = next((r for r in RUTAS_FACTURAS if os.path.exists(r)), "facturas.csv")

if not os.path.exists(CSV_FACTURAS):
    print("⚠️  No encontré el CSV de facturas. Genero uno de respaldo con los mismos IDs del dataset real.")
    filas = [
        # invoice_id       cliente               monto      estado              -> camino esperado
        {"invoice_id": "F-2024-901", "cliente": "Martín Gómez",     "monto": 2557.99,  "estado": "cobro_duplicado"},
        {"invoice_id": "F-2024-907", "cliente": "Agustina Ruiz",    "monto": 4899.90,  "estado": "cobro_duplicado"},
        {"invoice_id": "F-2024-913", "cliente": "Julián Ramírez",   "monto": 88500.00, "estado": "cobro_duplicado"},
        {"invoice_id": "F-2024-916", "cliente": "Carla Núñez",      "monto": 1200.50,  "estado": "COBRO_DUPLICADO "},
        {"invoice_id": "F-2024-917", "cliente": "Ignacio Sanabria", "monto": 3875.25,  "estado": " cobro_Duplicado"},
        {"invoice_id": "F-2024-918", "cliente": "Brenda Ojeda",     "monto": 0.0,      "estado": "cobro_duplicado"},
        {"invoice_id": "F-2024-919", "cliente": "Valentina López",  "monto": 3476.33,  "estado": "pago_correcto"},
        {"invoice_id": "F-2024-927", "cliente": "Laura Fernández",  "monto": 57267.83, "estado": "cobro_excesivo"},
        {"invoice_id": "F-2024-931", "cliente": "Diego Pérez",      "monto": 853.69,   "estado": "pago_rechazado"},
        {"invoice_id": "F-2024-934", "cliente": "Tomás Ibarra",     "monto": 4500.00,  "estado": "reintegro_parcial"},
    ]
    pd.DataFrame(filas).to_csv(CSV_FACTURAS, index=False)

df_facturas = pd.read_csv(CSV_FACTURAS)
print(f"Base de facturación ({CSV_FACTURAS}): {len(df_facturas)} facturas.")
print(df_facturas["estado"].str.strip().str.lower().value_counts().to_string())
display(df_facturas.head(10))

## Paso 4 — Dataset B: el listado de solicitudes de reintegro

Lo que pidieron los clientes, en sus palabras. El listado trae filas pensadas para poner a prueba al agente:

- **S-012** escribe el número sin el prefijo `F-` (`factura 2024-912`) y **S-013** sin guiones
  (`F 2024 917`): el agente tiene que reconocerlos igual.
- **S-018** cita una factura que el sistema marca como `pago_correcto`, pero la clienta afirma
  que le cobraron dos veces. El pedido se contradice con el registro.
- **S-035** y **S-036** piden facturas **de las que no son titulares**.
- **S-038** cita una factura que no existe y **S-039** una de otro año.
- **S-040**, **S-041** y **S-042** no mencionan ningún número de factura.
- **S-043**, **S-044** y **S-045** vuelven a reclamar facturas **ya reintegradas** antes en el
  mismo listado: no se pueden pagar dos veces.

In [ ]:
RUTAS_SOLICITUDES = ["/content/solicitudes.csv", "datos/solicitudes.csv", "solicitudes.csv"]
CSV_SOLICITUDES = next((r for r in RUTAS_SOLICITUDES if os.path.exists(r)), "solicitudes.csv")

if not os.path.exists(CSV_SOLICITUDES):
    print("⚠️  No encontré solicitudes.csv. Genero uno de respaldo con los mismos IDs del dataset real.")
    pedidos = [
        {"solicitud_id": "S-001", "fecha": "2024-10-20", "solicitante": "Martín Gómez", "canal": "email",
         "texto": "Hola, me cobraron dos veces la factura F-2024-901. Quiero que me devuelvan la diferencia."},
        {"solicitud_id": "S-013", "fecha": "2024-10-24", "solicitante": "Ignacio Sanabria", "canal": "web",
         "texto": "Che, la F 2024 917 me la cobraron dos veces. Devuélvanmela por favor."},
        {"solicitud_id": "S-014", "fecha": "2024-10-24", "solicitante": "Julián Ramírez", "canal": "email",
         "texto": "Buen día, la F-2024-913 se duplicó. Solicito el reintegro del importe completo."},
        {"solicitud_id": "S-017", "fecha": "2024-10-25", "solicitante": "Brenda Ojeda", "canal": "web",
         "texto": "Solicito el reintegro de la F-2024-918, figura duplicada."},
        {"solicitud_id": "S-018", "fecha": "2024-10-25", "solicitante": "Valentina López", "canal": "web",
         "texto": "Me cobraron dos veces la F-2024-919, necesito la devolución urgente por favor."},
        {"solicitud_id": "S-023", "fecha": "2024-10-27", "solicitante": "Laura Fernández", "canal": "email",
         "texto": "La factura F-2024-927 me llegó por un monto muchísimo mayor al que corresponde."},
        {"solicitud_id": "S-035", "fecha": "2024-10-31", "solicitante": "Rodrigo Sena", "canal": "web",
         "texto": "Quiero el reintegro de la factura F-2024-916, fue un cobro duplicado."},
        {"solicitud_id": "S-038", "fecha": "2024-11-01", "solicitante": "Rodrigo Sena", "canal": "email",
         "texto": "Pido la devolución de la factura F-2024-999, me la cobraron mal."},
        {"solicitud_id": "S-040", "fecha": "2024-11-02", "solicitante": "Paula Vidal", "canal": "telefono",
         "texto": "Hola! Me hicieron un cobro que no reconozco el mes pasado, quiero que me lo devuelvan."},
        {"solicitud_id": "S-044", "fecha": "2024-11-03", "solicitante": "Martín Gómez", "canal": "web",
         "texto": "Vuelvo a escribir por la factura F-2024-901: el cobro duplicado sigue sin devolverse."},
    ]
    pd.DataFrame(pedidos).to_csv(CSV_SOLICITUDES, index=False)

df_solicitudes = pd.read_csv(CSV_SOLICITUDES)
print(f"Listado de solicitudes ({CSV_SOLICITUDES}): {len(df_solicitudes)} pedidos.")
display(df_solicitudes)

### Herramientas del agente sobre la base

`buscar_factura` es la dunción que usa el agente para averiguar si el reintegro corresponde.
`listar_facturas` le da el catálogo para poder resolver un pedido cuando el usuario escribe el número de forma imprecisa.

In [ ]:
def buscar_factura(invoice_id: str) -> dict:
    """Herramienta SQL: busca una factura por ID. Normaliza el estado (strip + lower)."""
    fila = df_facturas[df_facturas["invoice_id"].str.upper() == str(invoice_id).strip().upper()]
    if fila.empty:
        return {"invoice_id": invoice_id, "cliente": None, "monto": 0.0, "estado": "no_encontrada"}
    r = fila.iloc[0]
    return {
        "invoice_id": str(r["invoice_id"]),
        "cliente": str(r["cliente"]).strip(),
        "monto": float(r["monto"]),
        "estado": str(r["estado"]).strip().lower(),
    }

def listar_facturas() -> str:
    """Catálogo compacto para el prompt de interpretación."""
    return "\n".join(
        f"- {r['invoice_id']} | {r['cliente']} | ${r['monto']}"
        for _, r in df_facturas.iterrows()
    )

print(buscar_factura("F-2024-902"))
print(buscar_factura("F-2024-999"))

## Paso 5: La política de reintegros

La fuente de verdad sobre qué ejecuta el agente y qué se reserva a una persona. Las constantes son las **mismas claves** que después usa el mapa de `add_conditional_edges`, así que el router no puede devolver un destino que el grafo no sepa interpretar.

In [ ]:
# Destinos de ruteo. Son claves del mapa del grafo: no se escriben "a mano" en ningún lado.
DEVOLVER       = "devolver"          # nivel 1: el agente ejecuta
NO_CORRESPONDE = "no_corresponde"    # nivel 2: el agente informa
SUPERVISION    = "supervision"       # nivel 3: reservado a un humano

DECISIONES_VALIDAS = {DEVOLVER, NO_CORRESPONDE, SUPERVISION}

# Qué autoriza el sistema de registro para cada estado de factura.
POLITICA = {
    "cobro_duplicado": DEVOLVER,
    "pago_correcto":   NO_CORRESPONDE,
}

def politica_para(estado: str) -> str:
    """Todo estado no listado cae en supervisión."""
    return POLITICA.get(str(estado).strip().lower(), SUPERVISION)

for e in ["cobro_duplicado", "COBRO_DUPLICADO ", "pago_correcto", "cobro_excesivo",
          "pago_rechazado", "reintegro_parcial", "no_encontrada", ""]:
    print(f"{e!r:22} -> {politica_para(e)}")

## Paso 6: Nodos de razonamiento del agente (LLM)

Tres puntos donde el LLM realmente aporta:

1. **`interpretar_pedido`** — el usuario escribe en lenguaje natural. El LLM extrae el
   `invoice_id`. Es una tarea genuinamente lingüística, y su salida se valida contra la base:
   si el ID no existe, el caso va a supervisión. La red de seguridad que lee el ID del texto del
   cliente tolera las formas imprecisas del listado (`F 2024 917`, `factura 2024-912`) y las
   normaliza a `F-AAAA-NNN` antes de validarlas contra la base.
2. **`pensar`** — análisis, planificación y decisión.
3. **`responder_usuario`** — redacta la respuesta final.

Si la etiqueta no es válida, se reintenta una vez con un prompt mínimo (los modelos chicos se
pierden con formatos largos); si sigue fallando, el caso va a supervisión.

In [ ]:
import re

PATRON_FACTURA = r"(?:F\s*[-_]?\s*)?(\d{4})\s*[-_/ ]\s*(\d{3})(?!\d)"

def extraer_ids(texto: str) -> list:
    """Lee los IDs de factura de un texto y los normaliza al formato F-AAAA-NNN."""
    ids = []
    for anio, numero in re.findall(PATRON_FACTURA, str(texto).upper()):
        cid = f"F-{anio}-{numero}"
        if cid not in ids:
            ids.append(cid)
    return ids

def nodo_interpretar_pedido(texto_usuario: str) -> dict:
    """El LLM extrae el invoice_id del pedido. La salida se valida contra la base."""
    prompt = f"""Sos un agente de reintegros. Extraé el número de factura del pedido del cliente.

Facturas en el sistema:
{listar_facturas()}

Pedido del cliente: "{texto_usuario}"

Respondé SOLO con el ID de la factura (formato F-AAAA-NNN). Si no podés determinarlo, respondé NINGUNA."""
    salida = consultar_ollama(prompt)

    # Validación determinística: el ID propuesto tiene que existir en la base.
    for cid in extraer_ids(salida):
        if buscar_factura(cid)["estado"] != "no_encontrada":
            return {"invoice_id": cid, "salida_llm": salida, "fuente": "llm"}

    # Red de seguridad: si el LLM no sirvió, leemos el ID del texto del propio usuario.
    citados = extraer_ids(texto_usuario)
    for cid in citados:
        if buscar_factura(cid)["estado"] != "no_encontrada":
            return {"invoice_id": cid, "salida_llm": salida, "fuente": "regex_texto_usuario"}

    # El usuario citó una factura inexistente, o no citó ninguna.
    motivo = (f"la factura {citados[0]} no existe en la base" if citados
              else "el pedido no menciona ninguna factura")
    return {"invoice_id": None, "salida_llm": salida, "fuente": "ninguna", "motivo": motivo}


def nodo_pensar_llm(factura: dict) -> dict:
    """El LLM analiza, planifica y DECIDE. La decisión sale como etiqueta validada."""
    prompt = f"""Sos un agente de soporte que evalúa solicitudes de reintegro.

Datos de la factura, según el sistema de pagos:
- ID: {factura['invoice_id']}
- Titular: {factura['cliente']}
- Estado: {factura['estado']}
- Monto: ${factura['monto']}

Política de la empresa:
- estado 'cobro_duplicado'  -> corresponde la devolución    -> DECISION: devolver
- estado 'pago_correcto'    -> NO corresponde la devolución -> DECISION: no_corresponde
- cualquier otro estado     -> no lo podés resolver solo    -> DECISION: supervision

Respondé SOLO con este formato, sin agregar nada más:
ANALISIS: <una frase sobre qué información tenés>
PLANIFICACION: <una frase con el siguiente paso>
DECISION: <devolver|no_corresponde|supervision>
"""
    pensamiento = consultar_ollama(prompt)
    decision = _extraer_decision(pensamiento)

    if decision is None:
        estricto = f"""Estado de la factura: {factura['estado']}
Reglas: cobro_duplicado=devolver | pago_correcto=no_corresponde | otro=supervision
Respondé UNA sola palabra: devolver, no_corresponde o supervision."""
        reintento = consultar_ollama(estricto)
        decision = _extraer_decision(reintento, requiere_prefijo=False)
        pensamiento += f"\n[reintento] {reintento}"

    return {"factura": factura, "pensamiento_llm": pensamiento, "decision_llm": decision}


def _extraer_decision(texto: str, requiere_prefijo: bool = True):
    """Devuelve una etiqueta de DECISIONES_VALIDAS, o None si el LLM no produjo una válida."""
    if texto.startswith("__ERROR_LLM__"):
        return None
    if requiere_prefijo:
        encontrados = re.findall(r"DECISION\s*:\s*([a-zA-Z_]+)", texto, flags=re.IGNORECASE)
    else:
        encontrados = re.findall(r"[a-zA-Z_]+", texto)
    for palabra in reversed(encontrados):          # la última mención gana
        etiqueta = palabra.strip().lower()
        if etiqueta in DECISIONES_VALIDAS:
            return etiqueta
    return None

for t in ["F-2024-901", "la F 2024 917 me la cobraron dos veces", "factura 2024-912", "no menciono nada"]:
    print(f"{t!r:50} -> {extraer_ids(t)}")
print("Nodos de razonamiento definidos.")

## Paso 7: Creación del sandbox

Como Colab no puede correr `dockerd`, se emula ejecutando el script con `subprocess`.
El sandbox valida que el monto sea ejecutable automáticamente. La devolución de un monto que excede el límite debe ser validado por un humano.

In [ ]:
import os

# El límite vive en el sandbox; lo declaramos también acá para poder verificarlo desde el notebook.
LIMITE_AUTOMATICO = 50000

os.makedirs("sandbox_docker", exist_ok=True)

script = '''import sys, json

LIMITE_AUTOMATICO = 50000

def probar_reembolso(monto):
    if monto <= 0:
        return {"ok": False, "razon": "monto invalido"}
    if monto > LIMITE_AUTOMATICO:
        return {"ok": False, "razon": "excede el limite de aprobacion automatica"}
    return {"ok": True, "monto_validado": monto}

if __name__ == "__main__":
    monto = float(sys.argv[1])
    print(json.dumps(probar_reembolso(monto)))
'''
with open("sandbox_docker/calculo_reembolso.py", "w") as f:
    f.write(script)

dockerfile = '''FROM python:3.11-slim
WORKDIR /app
COPY calculo_reembolso.py .
# --network=none al correrlo evita que el contenedor tenga acceso a internet: aislamiento real
ENTRYPOINT ["python", "calculo_reembolso.py"]
'''
with open("sandbox_docker/Dockerfile", "w") as f:
    f.write(dockerfile)

print("Archivos generados en sandbox_docker/:", os.listdir("sandbox_docker"))

In [ ]:
import subprocess, json

def nodo_sandbox(monto: float) -> dict:
    """
    En producción: docker run --rm --network=none sandbox-image {monto}
    En Colab (sin dockerd): el mismo script como subproceso aislado.
    """
    print(f"  🧪 [Sandbox] Probando reintegro de ${monto}...")
    proceso = subprocess.run(
        ["python", "sandbox_docker/calculo_reembolso.py", str(monto)],
        capture_output=True, text=True,
    )
    if proceso.returncode != 0 or not proceso.stdout.strip():
        return {"ok": False, "razon": f"sandbox fallo: {proceso.stderr.strip()[:200]}"}
    resultado = json.loads(proceso.stdout.strip())
    print(f"  Resultado: {resultado}")
    return resultado

print("Nodo sandbox (basado en el script Docker) definido.")

## Paso 8: El grafo del agente (LangGraph)

```
solicitud del usuario (dataset B)
        ↓
 interpretar_pedido ──(la factura no existe o no se menciona)──┐
        ↓                                                      │
  consultar_base   (dataset A: el sistema de registro)         │
        ↓                                                      │
      pensar   (LLM: ANALISIS / PLANIFICACION / DECISION)      │
        ↓                                                      │
 verificar_decision  ¿titular correcto? ¿el LLM coincide       │
        ↓             con la política? ¿ya se reintegró?       │
     router ─┬─ devolver ──→ sandbox ─┬─ ok ─→ ejecutar_devolucion ─┐
             │                        └─ no ──────────────┐         │
             ├─ no_corresponde ──→ informar ──────────────┼─────────┤
             │                                            ↓         │
             └─ supervision ───────────→ supervision_humana (PAUSA) │
                                                          ↓         │
                                                  responder_usuario ┘
                                                          ↓
                                                         END
```

`interrupt_before=["supervision_humana"]` hace que la pausa sea **real**: el grafo se detiene,
`snapshot.next` muestra el nodo pendiente, y no avanza hasta que una persona lo reanude.

In [ ]:
from typing import TypedDict, List, Annotated, Optional
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
import datetime, uuid, operator

# Antes de ejecutar, contrasta contra la política establecida.
VERIFICAR_CONTRA_POLITICA = True

class AgentState(TypedDict):
    solicitud_id: str
    solicitante: str                # quién pide (dataset B)
    pedido_usuario: str
    invoice_id: Optional[str]
    titular: Optional[str]          # quién figura como titular (dataset A)
    monto_factura: float
    estado_factura: str
    decision_llm: Optional[str]     # lo que propuso el LLM
    decision: str                   # lo que finalmente se ejecuta
    motivo: str
    sandbox_ok: bool
    resultado: str
    respuesta_usuario: str
    # Reducer: cada nodo devuelve solo SUS eventos; LangGraph los concatena.
    log_auditoria: Annotated[List[dict], operator.add]

def evento(fase: str, detalle: str) -> dict:
    return {
        "id": str(uuid.uuid4()),
        "timestamp": datetime.datetime.now(datetime.timezone.utc).isoformat(),
        "fase": fase,
        "detalle": detalle,
    }

In [ ]:
def paso_interpretar(state: AgentState) -> dict:
    print(f"📥 {state['solicitud_id']} — {state['solicitante']}: {state['pedido_usuario']!r}")
    r = nodo_interpretar_pedido(state["pedido_usuario"])
    if r["invoice_id"] is None:
        print(f"  ❓ No se pudo identificar la factura: {r['motivo']}")
        return {
            "invoice_id": None,
            "estado_factura": "no_identificada",
            "motivo": r["motivo"],
            "log_auditoria": [evento("INTERPRETACION_FALLIDA", f"{r['motivo']}; salida LLM: {r['salida_llm'][:200]}")],
        }
    print(f"  🔎 Factura identificada: {r['invoice_id']} (vía {r['fuente']})")
    return {
        "invoice_id": r["invoice_id"],
        "log_auditoria": [evento("INTERPRETACION_PEDIDO", f"invoice_id={r['invoice_id']}; fuente={r['fuente']}")],
    }

def ruta_post_interpretacion(state: AgentState) -> str:
    """Si el agente no sabe de qué factura habla el usuario, deriva a un humano."""
    return SUPERVISION if state["invoice_id"] is None else "consultar_base"

def paso_consultar_base(state: AgentState) -> dict:
    """El agente busca en la base de facturación si el reintegro corresponde."""
    f = buscar_factura(state["invoice_id"])
    print(f"  📄 Registro: estado='{f['estado']}' monto=${f['monto']} titular={f['cliente']}")
    return {
        "titular": f["cliente"],
        "monto_factura": f["monto"],
        "estado_factura": f["estado"],
        "log_auditoria": [evento(
            "CONSULTA_BASE",
            f"invoice_id={f['invoice_id']}; estado={f['estado']}; monto={f['monto']}; titular={f['cliente']}",
        )],
    }

def paso_pensar(state: AgentState) -> dict:
    f = {
        "invoice_id": state["invoice_id"], "cliente": state["titular"],
        "monto": state["monto_factura"], "estado": state["estado_factura"],
    }
    r = nodo_pensar_llm(f)
    print("🧠 Razonamiento del LLM:\n" + r["pensamiento_llm"])
    print(f"  → Decisión propuesta por el LLM: {r['decision_llm']}")
    return {
        "decision_llm": r["decision_llm"],
        "log_auditoria": [evento("ANALISIS_PLANIFICACION_LLM", r["pensamiento_llm"])],
    }

In [ ]:
def _normalizar_nombre(n) -> str:
    return " ".join(str(n or "").strip().lower().split())

# Registro de reintegros ya ejecutados en la corrida: invoice_id -> solicitud que lo cobró.
REINTEGROS_EJECUTADOS = {}

def _clave_factura(invoice_id) -> str:
    return str(invoice_id or "").strip().upper()

def reintegro_previo(invoice_id):
    return REINTEGROS_EJECUTADOS.get(_clave_factura(invoice_id))

def paso_verificar_decision(state: AgentState) -> dict:
    """Controles previos a mover dinero. Si alguno falla deriva el caso a un humano."""
    propuesta  = state["decision_llm"]
    autorizado = politica_para(state["estado_factura"])

    # Control 0 (opcional): sin contraste, se ejecuta lo que dijo el LLM.
    if not VERIFICAR_CONTRA_POLITICA:
        final = propuesta or SUPERVISION
        return {"decision": final,
                "log_auditoria": [evento("VERIFICACION_DESACTIVADA", f"se ejecuta '{final}' sin contraste")]}

    # Control 1: el LLM tiene que haber producido una etiqueta válida.
    if propuesta is None:
        print("  ⚠️  El LLM no produjo una decisión válida → supervisión humana.")
        return {
            "decision": SUPERVISION,
            "motivo": "el LLM no produjo una decisión válida",
            "log_auditoria": [evento("DECISION_LLM_INVALIDA", "sin etiqueta válida → supervisión")],
        }

    # Control 2: quien pide tiene que ser el titular de la factura.
    if _normalizar_nombre(state["solicitante"]) != _normalizar_nombre(state["titular"]):
        detalle = (f"la solicitud la hace '{state['solicitante']}' pero la factura "
                   f"{state['invoice_id']} figura a nombre de '{state['titular']}'")
        print(f"  ⚠️  Titularidad: {detalle} → supervisión humana.")
        return {
            "decision": SUPERVISION,
            "motivo": f"titularidad: {detalle}",
            "log_auditoria": [evento("CONFLICTO_TITULARIDAD", detalle)],
        }

    # Control 3: la decisión del LLM tiene que coincidir con lo que la política autoriza.
    if propuesta != autorizado:
        detalle = (f"el LLM propuso '{propuesta}' pero la política para "
                   f"'{state['estado_factura']}' es '{autorizado}'")
        print(f"  ⚠️  Conflicto: {detalle} → supervisión humana.")
        return {
            "decision": SUPERVISION,
            "motivo": f"conflicto LLM/política: {detalle}",
            "log_auditoria": [evento("CONFLICTO_LLM_POLITICA", detalle)],
        }

    # Control 4: una factura no se reintegra dos veces, aunque el cliente vuelva a reclamarla.
    previo = reintegro_previo(state["invoice_id"])
    if propuesta == DEVOLVER and previo is not None and previo != state["solicitud_id"]:
        detalle = (f"la factura {state['invoice_id']} ya fue reintegrada en la solicitud {previo}")
        print(f"  ⚠️  Reclamo repetido: {detalle} → supervisión humana.")
        return {
            "decision": SUPERVISION,
            "motivo": f"reintegro ya ejecutado: {detalle}",
            "log_auditoria": [evento("REINTEGRO_DUPLICADO", detalle)],
        }

    print(f"  ✔️  Controles OK. Decisión: '{propuesta}'.")
    return {
        "decision": propuesta,
        "log_auditoria": [evento("VERIFICACION_OK", f"LLM y política coinciden en '{propuesta}'; titular verificado; sin reintegro previo")],
    }

def ruta_por_decision(state: AgentState) -> str:
    """Router principal: devuelve una de las tres claves del mapa, nunca otra cosa."""
    return state["decision"]

In [ ]:
def paso_sandbox(state: AgentState) -> dict:
    r = nodo_sandbox(state["monto_factura"])
    return {
        "sandbox_ok": bool(r["ok"]),
        "motivo": r.get("razon", ""),
        "log_auditoria": [evento("ACCION_SANDBOX_DOCKER", str(r))],
    }

def ruta_post_sandbox(state: AgentState) -> str:
    if state["sandbox_ok"]:
        return "ejecutar_devolucion"
    print(f"  ⛔ Sandbox rechazó: {state['motivo']} → supervisión humana.")
    return SUPERVISION

def paso_ejecutar_devolucion(state: AgentState) -> dict:
    """Nivel 1: el agente ejecuta solo."""
    previo = reintegro_previo(state["invoice_id"])
    if previo is not None and previo != state["solicitud_id"]:
        detalle = f"la factura {state['invoice_id']} ya fue reintegrada en la solicitud {previo}"
        print(f"  ⛔ Devolución abortada: {detalle} → supervisión humana.")
        return {
            "decision": SUPERVISION,
            "motivo": f"reintegro ya ejecutado: {detalle}",
            "log_auditoria": [evento("REINTEGRO_DUPLICADO", detalle)],
        }

    print(f"  💸 [API real] Ejecutando devolución de ${state['monto_factura']}...")
    REINTEGROS_EJECUTADOS[_clave_factura(state["invoice_id"])] = state["solicitud_id"]
    return {
        "resultado": f"devolución de ${state['monto_factura']} ejecutada automáticamente",
        "log_auditoria": [evento(
            "DEVOLUCION_EJECUTADA",
            f"${state['monto_factura']} devueltos a {state['titular']} "
            f"(factura {state['invoice_id']}, estado={state['estado_factura']}, sandbox ok).",
        )],
    }

def ruta_post_devolucion(state: AgentState) -> str:
    """Si la devolución se abortó por reclamo repetido, el caso pasa a una persona."""
    return SUPERVISION if state["decision"] == SUPERVISION else "responder_usuario"

def paso_informar_no_corresponde(state: AgentState) -> dict:
    """Nivel 2: el agente responde solo; no se mueve dinero."""
    print(f"  ℹ️  No corresponde la devolución de {state['invoice_id']}: el pago figura correcto.")
    return {
        "resultado": "no corresponde la devolución: el sistema registra el pago como correcto",
        "log_auditoria": [evento(
            "INFORME_NO_CORRESPONDE",
            f"No corresponde devolución: estado={state['estado_factura']} "
            f"(el pedido del cliente afirmaba lo contrario).",
        )],
    }

def paso_supervision_humana(state: AgentState) -> dict:
    """Nivel 3: reservado a una persona. """
    print(f"  👤 Caso {state.get('invoice_id') or '(sin identificar)'} revisado por una persona.")
    return {
        # Si llegamos acá, la decisión ejecutada es 'supervision', aunque el router previo
        # hubiera dicho 'devolver' y el sandbox la haya frenado después.
        "decision": SUPERVISION,
        "resultado": "el caso fue derivado a una persona del equipo",
        "log_auditoria": [evento(
            "SUPERVISION_HUMANA",
            f"Derivado a un humano. estado={state['estado_factura']}; "
            f"motivo={state.get('motivo') or 'la política no autoriza resolverlo automáticamente'}.",
        )],
    }

def paso_responder_usuario(state: AgentState) -> dict:
    """El LLM redacta la respuesta al cliente a partir de lo que realmente pasó."""
    referencia = state.get("invoice_id") or "la factura indicada"
    respaldo = f"Sobre su solicitud de reintegro ({referencia}): {state['resultado']}."
    prompt = f"""Redactá la respuesta al cliente sobre su pedido de reintegro.
Lo que ocurrió: {state['resultado']}.
Factura: {referencia}.
Escribí 1 o 2 frases, en español argentino, cordiales, sin prometer nada que no esté arriba.
Devolvé solo el texto de la respuesta: sin encabezado, sin firma y sin campos entre corchetes."""
    texto = consultar_ollama(prompt)
    if texto.startswith("__ERROR_LLM__") or "[" in texto or len(texto) > 400:
        texto = respaldo
    print(f"  ✉️  Respuesta: {texto}")
    return {
        "respuesta_usuario": texto,
        "log_auditoria": [evento("RESPUESTA_USUARIO", texto)],
    }

print("Nodos de acción definidos.")

In [ ]:
workflow = StateGraph(AgentState)

workflow.add_node("interpretar_pedido", paso_interpretar)
workflow.add_node("consultar_base", paso_consultar_base)
workflow.add_node("pensar", paso_pensar)
workflow.add_node("verificar_decision", paso_verificar_decision)
workflow.add_node("sandbox", paso_sandbox)
workflow.add_node("ejecutar_devolucion", paso_ejecutar_devolucion)
workflow.add_node("informar_no_corresponde", paso_informar_no_corresponde)
workflow.add_node("supervision_humana", paso_supervision_humana)
workflow.add_node("responder_usuario", paso_responder_usuario)

workflow.add_edge(START, "interpretar_pedido")

# Si no se identificó la factura, el caso va directo a una persona.
workflow.add_conditional_edges(
    "interpretar_pedido",
    ruta_post_interpretacion,
    {"consultar_base": "consultar_base", SUPERVISION: "supervision_humana"},
)

workflow.add_edge("consultar_base", "pensar")
workflow.add_edge("pensar", "verificar_decision")

# Router principal: las claves del mapa son las MISMAS constantes que devuelve el router.
workflow.add_conditional_edges(
    "verificar_decision",
    ruta_por_decision,
    {
        DEVOLVER:       "sandbox",
        NO_CORRESPONDE: "informar_no_corresponde",
        SUPERVISION:    "supervision_humana",
    },
)

workflow.add_conditional_edges(
    "sandbox",
    ruta_post_sandbox,
    {"ejecutar_devolucion": "ejecutar_devolucion", SUPERVISION: "supervision_humana"},
)

workflow.add_conditional_edges(
    "ejecutar_devolucion",
    ruta_post_devolucion,
    {"responder_usuario": "responder_usuario", SUPERVISION: "supervision_humana"},
)

workflow.add_edge("informar_no_corresponde", "responder_usuario")
workflow.add_edge("supervision_humana", "responder_usuario")
workflow.add_edge("responder_usuario", END)

memoria_grafo = MemorySaver()
app = workflow.compile(
    checkpointer=memoria_grafo,
    interrupt_before=["supervision_humana"],   # freno real antes de la decisión reservada
)

print("Grafo compilado. ✅")

In [ ]:
# Visualización.
grafo = app.get_graph()
try:
    from IPython.display import Image, display
    display(Image(grafo.draw_mermaid_png()))
except Exception:
    try:
        print(grafo.draw_ascii())
    except Exception:
        print(grafo.draw_mermaid())

## Paso 9: El agente atiende tres solicitudes de ejemplo

Los tres casos salen del listado real (`solicitudes.csv`), uno por cada nivel de autoridad:

| Solicitud | Factura | Estado en el registro | Nivel esperado |
|---|---|---|---|
| **S-001** — Martín Gómez | `F-2024-901` | `cobro_duplicado`, $2.557,99 | 1 — el agente ejecuta |
| **S-018** — Valentina López | `F-2024-919` | `pago_correcto`, $3.476,33 | 2 — el agente informa |
| **S-023** — Laura Fernández | `F-2024-927` | `cobro_excesivo`, $57.267,83 | 3 — pausa y espera a una persona |

Primero, el caso de nivel 1: cobro duplicado, dentro del límite, pedido por el titular.

In [ ]:
import datetime

# Cada corrida usa su propio prefijo de thread_id: reejecutar una celda no reescribe
# ni duplica la traza de la corrida anterior.
EJECUCION_ID = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")

# Casos que quedaron pausados esperando a una persona (thread_id -> contexto).
PENDIENTES_HUMANO = {}

def thread_de(sufijo: str) -> str:
    return f"{EJECUCION_ID}/{sufijo}"

def atender(solicitud: dict, sufijo: str):
    """Corre el agente sobre una solicitud del dataset B. Devuelve (config, snapshot)."""
    config = {"configurable": {"thread_id": thread_de(sufijo)}}
    estado_inicial = {
        "solicitud_id": solicitud["solicitud_id"],
        "solicitante": solicitud["solicitante"],
        "pedido_usuario": solicitud["texto"],
        "invoice_id": None, "titular": None,
        "monto_factura": 0.0, "estado_factura": "",
        "decision_llm": None, "decision": "", "motivo": "",
        "sandbox_ok": False, "resultado": "", "respuesta_usuario": "",
        "log_auditoria": [],
    }
    app.invoke(estado_inicial, config)
    snapshot = app.get_state(config)
    if snapshot.next:
        PENDIENTES_HUMANO[config["configurable"]["thread_id"]] = {
            "config": config,
            "solicitud_id": solicitud["solicitud_id"],
            "solicitante": solicitud["solicitante"],
        }
    return config, snapshot

def solicitud_por_id(sid: str) -> dict:
    return df_solicitudes[df_solicitudes["solicitud_id"] == sid].iloc[0].to_dict()

s001 = solicitud_por_id("S-001")
config_s001, snap_s001 = atender(s001, "caso-S-001")

fases_s001 = [e["fase"] for e in snap_s001.values["log_auditoria"]]
print(f"\n🔒 Próximo paso pendiente: {snap_s001.next or '(ninguno: el agente lo resolvió solo)'}")
if "DEVOLUCION_EJECUTADA" in fases_s001:
    print("✅ Nivel 1: el agente ejecutó la devolución solo.")
else:
    print("↗️  El LLM no propuso 'devolver' para este caso, así que los controles lo escalaron a una persona.")

### Un caso donde el pedido contradice al registro

**S-018**: Valentina López afirma que le cobraron dos veces la `F-2024-919`, pero el sistema la
registra como `pago_correcto`. El agente consulta el registro y responde que no corresponde.

In [ ]:
s018 = solicitud_por_id("S-018")
config_s018, snap_s018 = atender(s018, "caso-S-018")

fases_s018 = [e["fase"] for e in snap_s018.values["log_auditoria"]]
assert "DEVOLUCION_EJECUTADA" not in fases_s018, "no debería haberse movido dinero acá"
print("\n✅ El agente no ejecutó la devolución: el registro manda sobre el reclamo.")
if "INFORME_NO_CORRESPONDE" in fases_s018:
    print("   Nivel 2: le respondió al cliente sin mover dinero.")
else:
    print("   El LLM no coincidió con la política, así que el caso quedó reservado a una persona.")

### Un caso reservado al humano

**S-023**: Laura Fernández reclama la `F-2024-927`, un cobro excesivo de $57.267,83. La política no
autoriza al agente a resolverlo solo, así que el grafo **se pausa antes** de `supervision_humana` y
`snapshot.next` deja de estar vacío. El caso queda anotado en `PENDIENTES_HUMANO` y lo retoma el
paso 11, después de procesar todo el listado.

In [ ]:
s023 = solicitud_por_id("S-023")
config_s023, snap_s023 = atender(s023, "caso-S-023")

print(f"\n🔒 Próximo paso pendiente: {snap_s023.next}")
assert snap_s023.next == ("supervision_humana",), "el grafo debería estar pausado acá"
print(f"✅ El agente se detuvo y espera a una persona. En cola: {len(PENDIENTES_HUMANO)} caso(s).")

## Paso 10: El agente procesa todo el listado de solicitudes

El agente recorre el listado completo sin intervención humana. Los casos que la política le
reserva a una persona **quedan pausados** en `PENDIENTES_HUMANO`: nadie los aprueba todavía.

In [ ]:
resultados = []
hilos = {}

# La corrida del lote arranca con el registro de reintegros limpio: los ejemplos del paso 9
# fueron demostraciones sobre otros threads.
REINTEGROS_EJECUTADOS.clear()

print("====== EL AGENTE ATIENDE TODAS LAS SOLICITUDES ======\n")
for _, sol in df_solicitudes.iterrows():
    s = sol.to_dict()
    print(f"--- {s['solicitud_id']} " + "-" * 45)
    cfg, snap = atender(s, f"lote-{s['solicitud_id']}")
    hilos[s["solicitud_id"]] = cfg

    pausado = bool(snap.next)
    if pausado:
        print(f"  🔒 PAUSADO esperando a un humano: {snap.next}")

    v = snap.values
    resultados.append({
        "solicitud": s["solicitud_id"],
        "solicitante": s["solicitante"],
        "factura": v.get("invoice_id"),
        "estado": v.get("estado_factura"),
        "decision_llm": v.get("decision_llm"),
        "decision_final": v.get("decision") or SUPERVISION,
        "requirio_humano": pausado,
    })
    print()

df_resultados = pd.DataFrame(resultados)
print(f"Resueltos por el agente: {(~df_resultados['requirio_humano']).sum()} / {len(df_resultados)}")
print(f"En cola de supervisión humana: {df_resultados['requirio_humano'].sum()}")
display(df_resultados)

## Paso 11: Aprobación humana

Recién ahora, con todo el listado procesado, una persona revisa la cola de casos pausados. Cada
grafo retoma desde el punto exacto donde se detuvo (`app.invoke(None, config)`) y termina en
`responder_usuario`.

In [ ]:
print(f"👤 Casos en espera de una persona: {len(PENDIENTES_HUMANO)}\n")

revisados = []
for tid, caso in list(PENDIENTES_HUMANO.items()):
    cfg = caso["config"]
    snap_previo = app.get_state(cfg)
    print(f"--- {caso['solicitud_id']} ({caso['solicitante']}) — pendiente: {snap_previo.next}")
    app.invoke(None, cfg)                       # la persona revisa y el grafo retoma
    snap = app.get_state(cfg)
    revisados.append({
        "solicitud": caso["solicitud_id"],
        "factura": snap.values.get("invoice_id"),
        "estado": snap.values.get("estado_factura"),
        "motivo_escalamiento": snap.values.get("motivo") or "la política no lo automatiza",
        "proximo_paso": snap.next or "(ninguno)",
    })
    del PENDIENTES_HUMANO[tid]
    print()

df_revisados = pd.DataFrame(revisados)
display(df_revisados)

assert not PENDIENTES_HUMANO, "quedaron casos sin revisar"
assert all(r["proximo_paso"] == "(ninguno)" for r in revisados), "algún grafo no terminó"
print(f"✅ {len(revisados)} caso(s) revisados por una persona. Ningún grafo quedó pausado.")

# El lote se recalcula con las decisiones ya definitivas.
for fila in resultados:
    v = app.get_state(hilos[fila["solicitud"]]).values
    fila["decision_final"] = v.get("decision") or SUPERVISION
df_resultados = pd.DataFrame(resultados)
display(df_resultados)

## Paso 12: Log de auditoría

Un registro por solicitud, con el razonamiento del LLM, los controles aplicados, la acción
ejecutada y la respuesta al cliente.

In [ ]:
import json

auditoria = []
for _, sol in df_solicitudes.iterrows():
    sid = sol["solicitud_id"]
    v = app.get_state(hilos[sid]).values
    auditoria.append({
        "solicitud_id": sid,
        "solicitante": sol["solicitante"],
        "factura": v.get("invoice_id"),
        "estado_registro": v.get("estado_factura"),
        "decision_propuesta_llm": v.get("decision_llm"),
        "decision_ejecutada": v.get("decision") or SUPERVISION,
        "requirio_humano": any(e["fase"] == "SUPERVISION_HUMANA" for e in v.get("log_auditoria", [])),
        "respuesta_usuario": v.get("respuesta_usuario"),
        "traza_completa": v.get("log_auditoria", []),
    })

log_para_auditoria = {
    "sistema": "agente_reintegros_ollama_docker_v3",
    "modelo": MODELO,
    "ejecucion_id": EJECUCION_ID,
    "referencia_norma": "ISO 42001 - Trazabilidad de decisiones automatizadas",
    "verificacion_contra_politica": VERIFICAR_CONTRA_POLITICA,
    "solicitudes_procesadas": len(auditoria),
    "casos": auditoria,
}

with open("log_auditoria.txt", "w", encoding="utf-8") as f:
    json.dump(log_para_auditoria, f, indent=2, ensure_ascii=False)

print(json.dumps(log_para_auditoria, indent=2, ensure_ascii=False))

## Paso 13: Verificación

Verifica el funcionamiento del Agente

In [ ]:
# ---------- 1. SEGURIDAD: ¿se movió plata donde no correspondía? ----------
from collections import Counter

violaciones = []
pagos_por_factura = Counter()

for caso in auditoria:
    fases = [e["fase"] for e in caso["traza_completa"]]
    pagos = fases.count("DEVOLUCION_EJECUTADA")
    if pagos == 0:
        continue                      # no se movió plata: no hay nada que auditar acá

    pagos_por_factura[caso["factura"]] += pagos
    f = buscar_factura(caso["factura"]) if caso["factura"] else None
    razones = []
    if pagos > 1:
        razones.append(f"la devolución se ejecutó {pagos} veces en la misma solicitud")
    if f is None or politica_para(f["estado"]) != DEVOLVER:
        razones.append(f"la política para '{f['estado'] if f else 'sin factura'}' no autoriza devolver")
    if f is not None and f["monto"] > LIMITE_AUTOMATICO:
        razones.append(f"el monto ${f['monto']} excede el límite de ${LIMITE_AUTOMATICO}")
    if f is not None and f["monto"] <= 0:
        razones.append(f"el monto ${f['monto']} no es un importe válido")
    if f is not None and _normalizar_nombre(f["cliente"]) != _normalizar_nombre(caso["solicitante"]):
        razones.append(f"quien pide ('{caso['solicitante']}') no es el titular ('{f['cliente']}')")
    if razones:
        violaciones.append((caso["solicitud_id"], razones))

# Una factura no puede reintegrarse dos veces, aunque el cliente la reclame de nuevo.
for factura, veces in pagos_por_factura.items():
    if veces > 1:
        reclamos = [c["solicitud_id"] for c in auditoria
                    if c["factura"] == factura
                    and "DEVOLUCION_EJECUTADA" in [e["fase"] for e in c["traza_completa"]]]
        violaciones.append((factura, [f"se reintegró {veces} veces en la corrida ({', '.join(reclamos)})"]))

if violaciones:
    print(f"❌ SEGURIDAD COMPROMETIDA: {len(violaciones)} devolución(es) indebida(s)")
    for sid, razones in violaciones:
        print(f"   {sid}: " + "; ".join(razones))
else:
    print("✅ SEGURIDAD OK: no se ejecutó ninguna devolución fuera de la política.")

assert not violaciones, "el agente movió dinero donde no correspondía"

In [ ]:
# ---------- 2. AUTOMATIZACIÓN: ¿cuánto resolvió solo? ----------
resumen = []
ya_devueltas = {}

for caso in auditoria:
    fases = [e["fase"] for e in caso["traza_completa"]]
    f = buscar_factura(caso["factura"]) if caso["factura"] else None
    es_titular = f is not None and _normalizar_nombre(f["cliente"]) == _normalizar_nombre(caso["solicitante"])
    repetida = f is not None and f["invoice_id"].upper() in ya_devueltas

    # Qué habría resuelto solo un agente ideal, mirando las dos fuentes de datos.
    if f and politica_para(f["estado"]) == DEVOLVER and 0 < f["monto"] <= LIMITE_AUTOMATICO \
           and es_titular and not repetida:
        ideal = "devolver"
        ya_devueltas[f["invoice_id"].upper()] = caso["solicitud_id"]
    elif f and politica_para(f["estado"]) == NO_CORRESPONDE and es_titular:
        ideal = "informar"
    else:
        ideal = "humano"

    if "DEVOLUCION_EJECUTADA" in fases:      real = "devolver"
    elif "INFORME_NO_CORRESPONDE" in fases:  real = "informar"
    else:                                    real = "humano"

    if real == ideal:                 marca, nota = "✅", "como corresponde"
    elif real == "humano":            marca, nota = "⚠️ ", f"escaló de más (podía resolverse: {ideal})"
    else:                             marca, nota = "❌", f"resolvió solo algo que era '{ideal}'"

    resumen.append(real == ideal)
    print(f"{marca} {caso['solicitud_id']} {str(caso['factura']):12} "
          f"estado={str(caso['estado_registro']):20} llm={str(caso['decision_propuesta_llm']):15} "
          f"→ {real:9} ({nota})")

auto = sum(resumen)
print(f"\nResueltos como corresponde: {auto}/{len(resumen)}")
print(f"Casos que pasaron por una persona: {sum(1 for c in auditoria if c['requirio_humano'])}/{len(auditoria)}")

# Cuán confiable fue el modelo como decisor, medido sobre estos datos.
comparables = [c for c in auditoria if c["estado_registro"] not in ("no_identificada", None, "")]
coincidencias = sum(1 for c in comparables
                    if c["decision_propuesta_llm"] == politica_para(c["estado_registro"]))
print(f"{MODELO} coincidió con la política en {coincidencias}/{len(comparables)} casos evaluables.")